In [ ]:
import nibabel as nib
import numpy as np
import os
import random
import math
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
root_dir = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_1"
demo_data = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_1_demographic.xlsx"
discs = os.listdir(root_dir)

paths = []

In [ ]:
df = pd.read_excel(demo_data)[['ID', 'CDR']]
df = df.dropna()
print(df)

In [ ]:
id_names = df['ID'].values
print(id_names)

In [ ]:
train_labels, val_labels = [], []

for sub in id_names:
    # print(sub)
    print(df[df['ID'] == sub]["ID"])

In [ ]:
sub_names = df['ID'].values

In [ ]:
mri_data = []
backup = []

for disc in discs:
    disc_dir = os.path.join(root_dir, disc)
    subjects = os.listdir(disc_dir)

    for sub in subjects:
        if sub in sub_names:
            sub_dir = os.path.join(disc_dir, sub, 'FSL_SEG')
            sub_dir_files = os.listdir(sub_dir)
            

            for item in sub_dir_files:
                if item.endswith(".img") and "MR2" not in item:
                    backup.append(os.path.join(sub_dir, item))
            if backup != []:
                mri_data.append(backup)
            backup = []

In [ ]:
n = len(mri_data)
print(n)

index = [i for i in range(n)]
random.shuffle(index)
print(index)

In [ ]:
for item in mri_data:
    print(item)

In [ ]:
train_part, val_part = 0.7*n, 0.3*n
train_data, val_data = [], []

count = 0

for i in range(n):
    if count < train_part:
        train_data.append(mri_data[i])
    else:
        val_data.append(mri_data[i])
    count += 1

print(f"TAMANHO DE CADA CONJUNTO\nTREINO: {len(train_data)}\nVALIDAÇÃO: {len(val_data)}\nTOTAL: {len(train_data + val_data)}")

In [ ]:
print(train_data[0][0])

img = nib.load(train_data[0][0])

plt.imshow(img.get_fdata()[:, :, 80], cmap='grey')

In [ ]:
print(df['ID'])

In [ ]:
import pandas as pd
import re

train_cdr_counts = [0, 0, 0, 0]
val_cdr_counts = [0, 0, 0, 0]

def count_cdr_labels(data_list, dataframe, counts_list):
    for item in data_list:
        filepath_string = item[0] 
        
        match = re.search(r'(OAS1_\d{4}_MR\d)', filepath_string)
        
        if match:
            subject_id = match.group(0)

            filtered_series = dataframe[dataframe['ID'] == subject_id]["CDR"]
            
            if not filtered_series.empty:
                cdr_value = filtered_series.iloc[0] 
            
                if cdr_value == 0.0:
                    counts_list[0] += 1
                elif cdr_value == 0.5:
                    counts_list[1] += 1
                elif cdr_value == 1.0:
                    counts_list[2] += 1
                elif cdr_value == 2.0:
                    counts_list[3] += 1
                    
    return counts_list

train_cdr_counts = count_cdr_labels(train_data, df, train_cdr_counts)

val_cdr_counts = count_cdr_labels(val_data, df, val_cdr_counts)

print("\n" + "="*45)
print("Mapeamento: [0.0, 0.5, 1.0, 2.0]")
print("-" * 45)
print(f"Contagem (TREINAMENTO): {train_cdr_counts}")
print(f"Contagem (VALIDAÇÃO): {val_cdr_counts}")
print("="*45)

In [ ]:
import pandas as pd
import re
import os
import shutil
import nibabel as nib # Importa a biblioteca NiBabel

# Definição da Estrutura de Pastas e Mapeamento
OUTPUT_ROOT = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_1_FSL_SEP"

CDR_FOLDER_MAP = {
    0.0: "0.0",
    0.5: "0.5",
    1.0: "1.0",
    2.0: "2.0"
}

def create_output_structure():
    print("Criando estrutura de pastas...")
    for subset in ["train", "validation"]:
        for cdr_folder in CDR_FOLDER_MAP.values():
            path = os.path.join(OUTPUT_ROOT, subset, cdr_folder)
            os.makedirs(path, exist_ok=True)
    print("Estrutura criada com sucesso.")

# --- FUNÇÃO MODIFICADA ---
def organize_files_by_cdr(data_list, dataframe, subset_name):
    files_saved = 0
    ids_not_found = 0
    
    for item in data_list:
        filepath_string = item[0] 
        
        match = re.search(r'(OAS1_\d{4}_MR\d)', filepath_string)
        
        if match:
            subject_id = match.group(0)
            
            filtered_series = dataframe[dataframe['ID'] == subject_id]["CDR"]
            
            if not filtered_series.empty:
                cdr_value = filtered_series.iloc[0] 
                
                if cdr_value in CDR_FOLDER_MAP:
                    cdr_folder_name = CDR_FOLDER_MAP[cdr_value]
                    
                    dest_dir = os.path.join(OUTPUT_ROOT, subset_name, cdr_folder_name)
                    
                    # ----------------------------------------------------
                    # PASSO 1: Carregar o arquivo de imagem
                    try:
                        img = nib.load(filepath_string)
                    except Exception as e:
                        print(f"Erro ao carregar {filepath_string}: {e}")
                        continue
                    
                    # PASSO 2: Definir o nome do novo arquivo NIfTI
                    # Ex: 'OAS1_0001_MR1.nii.gz'
                    new_filename = f"{subject_id}.nii.gz"
                    dest_path = os.path.join(dest_dir, new_filename)
                    
                    # PASSO 3: Salvar como NIfTI comprimido (.nii.gz)
                    nib.save(img, dest_path)
                    files_saved += 1
                    # ----------------------------------------------------
                
            else:
                ids_not_found += 1
                
    print(f"[{subset_name.upper()}]: {files_saved} arquivos NIfTI salvos.")
    print(f"[{subset_name.upper()}]: {ids_not_found} IDs não encontrados no DataFrame 'df'.")

# Execução

create_output_structure()

organize_files_by_cdr(train_data, df, "train")

organize_files_by_cdr(val_data, df, "validation")

print("\nProcesso de organização e conversão concluído! Os arquivos estão em NIfTI na pasta:", OUTPUT_ROOT)